# Progress Bar

> Compact segmented progress bar for multi-step workflows.

In [ ]:
#| default_exp components.progress_bar

In [ ]:
#| export
from dataclasses import dataclass
from typing import Callable, Optional, Sequence

from fasthtml.common import Div, FT

from cjm_fasthtml_tailwind.utilities.sizing import w, h
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex, flex_direction, flex_display
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.utilities.transitions_and_animation import transition, duration
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, border_dui
from cjm_fasthtml_daisyui.utilities.border_radius import border_radius
from cjm_fasthtml_daisyui.components.feedback.tooltip import tooltip, tooltip_placement

from cjm_fasthtml_step_progress.core.models import StepInfo
from cjm_fasthtml_step_progress.core.config import StepProgressConfig

## _build_segment_fill_cls

In [ ]:
#| export
def _build_segment_fill_cls(
    idx: int,          # segment index
    current: int,      # current step index
    config: StepProgressConfig,  # rendering config
) -> str:              # combined CSS class string for the fill div
    """Build CSS classes for a segment's inner fill div."""
    base = combine_classes(h.full, transition.colors, duration('200'))
    if idx < current:
        return combine_classes(base, getattr(bg_dui, config.fill_bg))
    elif idx == current:
        return combine_classes(base, getattr(bg_dui, config.fill_bg).opacity(config.current_opacity))
    else:
        return str(base)

In [ ]:
# Test _build_segment_fill_cls
cfg = StepProgressConfig()

# Completed segment
cls = _build_segment_fill_cls(0, 2, cfg)
assert 'bg-primary' in cls
assert '/60' not in cls
assert 'transition-colors' in cls

# Current segment
cls = _build_segment_fill_cls(2, 2, cfg)
assert 'bg-primary/60' in cls

# Future segment
cls = _build_segment_fill_cls(3, 2, cfg)
assert 'bg-primary' not in cls
assert 'h-full' in cls

# Custom fill color
custom_cfg = StepProgressConfig(fill_bg='secondary', current_opacity=40)
cls = _build_segment_fill_cls(1, 1, custom_cfg)
assert 'bg-secondary/40' in cls

print("_build_segment_fill_cls tests passed!")

## _build_segment

In [ ]:
#| export
def _build_segment(
    step: StepInfo,    # step descriptor
    idx: int,          # segment index
    current: int,      # current step index
    config: StepProgressConfig,  # rendering config
) -> FT:               # FastHTML element for one segment
    """Build a single progress bar segment."""
    fill_cls = _build_segment_fill_cls(idx, current, config)
    
    # Separator border on all segments after the first
    sep_cls = combine_classes(
        border.l('1'), getattr(border_dui, config.separator_color)
    ) if idx > 0 else ''
    
    if config.show_tooltips:
        # Tooltip wrapper + inner fill div
        tip_pos = getattr(tooltip_placement, config.tooltip_position)
        wrapper_cls = combine_classes(tooltip, tip_pos, flex('1'), sep_cls)
        return Div(
            Div(cls=fill_cls),
            cls=wrapper_cls,
            data_tip=step.title,
        )
    else:
        # Single div, no tooltip
        segment_cls = combine_classes(flex('1'), h.full, fill_cls, sep_cls)
        return Div(cls=segment_cls)

In [ ]:
from fasthtml.common import to_xml

# Test _build_segment with tooltips
cfg = StepProgressConfig()
step = StepInfo("Selection")

seg = _build_segment(step, 0, 1, cfg)
html = to_xml(seg)
assert 'data-tip="Selection"' in html
assert 'tooltip' in html
assert 'tooltip-top' in html
assert 'flex-1' in html
assert 'bg-primary' in html
assert 'border-l' not in html  # first segment, no separator

# Second segment has separator
seg2 = _build_segment(StepInfo("Review"), 1, 0, cfg)
html2 = to_xml(seg2)
assert 'border-l-1' in html2
assert 'border-base-100' in html2

# Test without tooltips
no_tip_cfg = StepProgressConfig(show_tooltips=False)
seg3 = _build_segment(step, 0, 0, no_tip_cfg)
html3 = to_xml(seg3)
assert 'tooltip' not in html3
assert 'data-tip' not in html3
assert 'flex-1' in html3

print("_build_segment tests passed!")

_build_segment tests passed!


## render_step_progress

In [ ]:
#| export
def render_step_progress(
    steps: Sequence[StepInfo],              # ordered step definitions
    current_index: int,                     # 0-based index of the current step
    config: Optional[StepProgressConfig] = None,  # rendering config (defaults used if None)
) -> FT:                                    # the progress bar element
    """Render a compact segmented progress bar showing position in a multi-step workflow."""
    if config is None:
        config = StepProgressConfig()
    
    # Handle empty steps
    if not steps:
        return Div(id=config.id)
    
    # Clamp current_index to valid range
    current_index = max(0, min(current_index, len(steps) - 1))
    
    # Build track container classes
    track_cls = combine_classes(
        flex_display,
        flex_direction.row,
        w.full,
        h(config.height),
        getattr(border_radius, config.border_radius),
        getattr(bg_dui, config.track_bg),
    )
    
    # Build segments
    segments = [
        _build_segment(step, idx, current_index, config)
        for idx, step in enumerate(steps)
    ]
    
    return Div(*segments, cls=track_cls, id=config.id)

In [ ]:
# Test render_step_progress
steps = [
    StepInfo("Selection"),
    StepInfo("Segment & Align"),
    StepInfo("Review"),
    StepInfo("Verify"),
]

# Default config, step 1
bar = render_step_progress(steps, 1)
html = to_xml(bar)
assert 'id="step-progress"' in html
assert 'flex' in html
assert 'flex-row' in html
assert 'w-full' in html
assert 'h-3' in html
assert 'rounded-field' in html
assert 'bg-base-300' in html
assert html.count('tooltip') >= 4  # 4 tooltip wrappers
assert 'data-tip="Selection"' in html
assert 'data-tip="Verify"' in html

print("render_step_progress basic test passed!")

In [ ]:
# Test edge cases

# Empty steps
empty_bar = render_step_progress([], 0)
html = to_xml(empty_bar)
assert 'id="step-progress"' in html
assert 'tooltip' not in html

# Out of range index (clamped)
bar = render_step_progress(steps, 10)
html = to_xml(bar)
assert 'tooltip' in html  # still renders

bar = render_step_progress(steps, -5)
html = to_xml(bar)
assert 'tooltip' in html  # still renders

# Single step
single = render_step_progress([StepInfo("Only Step")], 0)
html = to_xml(single)
assert 'data-tip="Only Step"' in html
assert 'border-l' not in html  # no separator for single step

# Custom config
custom = StepProgressConfig(
    fill_bg='accent', track_bg='base_200',
    border_radius='box', id='my-progress'
)
bar = render_step_progress(steps, 2, custom)
html = to_xml(bar)
assert 'id="my-progress"' in html
assert 'bg-base-200' in html
assert 'rounded-box' in html
assert 'bg-accent' in html

print("Edge case tests passed!")

Edge case tests passed!


## create_step_progress_renderer

In [ ]:
#| export
def create_step_progress_renderer(
    config: Optional[StepProgressConfig] = None,  # rendering config
) -> Callable:  # callable with signature (steps, current_index) -> FT
    """Create a progress renderer callback for StepFlow integration."""
    if config is None:
        config = StepProgressConfig()
    
    def _render(
        steps: Sequence[StepInfo],  # ordered step definitions
        current_index: int,         # 0-based current step index
    ) -> FT:                        # the progress bar element
        """Render step progress bar."""
        return render_step_progress(steps, current_index, config)
    
    return _render

In [ ]:
# Test create_step_progress_renderer
renderer = create_step_progress_renderer()
bar = renderer(steps, 0)
html = to_xml(bar)
assert 'id="step-progress"' in html
assert 'bg-primary' in html

# With custom config
custom_renderer = create_step_progress_renderer(
    StepProgressConfig(fill_bg='secondary')
)
bar = custom_renderer(steps, 1)
html = to_xml(bar)
assert 'bg-secondary' in html

print("create_step_progress_renderer tests passed!")

create_step_progress_renderer tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()